In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS olist_project.gold")

DataFrame[]

In [0]:
dim_customer = spark.table("olist_project.silver.customers")
dim_customer.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.gold.dim_customer")

In [0]:
category_translation = spark.table("olist_project.bronze.category_translation")
products_silver = spark.table("olist_project.silver.products")

dim_product = (products_silver
    .join(category_translation,
          products_silver.product_category_name
          == category_translation.product_category_name,
          "left")
    .select("product_id", "product_category_name_english",
            "product_weight_g", "product_length_cm",
            "product_height_cm", "product_width_cm")
)

dim_product.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.gold.dim_product")

In [0]:
dim_seller = spark.table("olist_project.silver.sellers")
dim_seller.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.gold.dim_seller")

In [0]:
from pyspark.sql.functions import explode, sequence, to_date, year, month, dayofmonth, date_format

date_range = spark.sql("""
    SELECT explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) as date
""")

dim_date = (date_range
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day_name", date_format("date", "EEEE"))
)
dim_date.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.gold.dim_date")

In [0]:
from pyspark.sql.functions import sum as _sum, countDistinct

orders_silver = spark.table("olist_project.silver.orders")
items_silver = spark.table("olist_project.silver.order_items")
payments_silver = spark.table("olist_project.silver.order_payments")

items_agg = (items_silver
    .groupBy("order_id")
    .agg(_sum("item_price").alias("total_item_price"),
         _sum("freight_value").alias("total_freight"),
         countDistinct("product_id").alias("distinct_products"))
)

payments_agg = (payments_silver
    .groupBy("order_id")
    .agg(_sum("payment_value").alias("total_paid"))
)

fact_orders = (orders_silver
    .join(items_agg, "order_id", "left")
    .join(payments_agg, "order_id", "left")
    .select("order_id", "customer_id", "order_status",
            "purchase_ts", "delivered_customer_ts", "estimated_delivery_ts",
            "total_item_price", "total_freight", "distinct_products", "total_paid")
)

fact_orders.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.gold.fact_orders")

print(f"fact_orders rows: {fact_orders.count()}")

fact_orders rows: 99441


In [0]:
from pyspark.sql.functions import to_date as _to_date, col as _col

daily_revenue = (fact_orders
    .withColumn("order_date", _to_date("purchase_ts"))
    .join(spark.table("olist_project.gold.dim_customer"), "customer_id")
    .groupBy("order_date", "state")
    .agg(_sum("total_paid").alias("revenue"),
         countDistinct("order_id").alias("order_count"))
)
daily_revenue.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.gold.daily_revenue_by_state")